In [ ]:
# %% [code] {"jupyter":{"outputs_hidden":false}}
import os
import pandas as pd
import kagglehub

# 1. Download latest version
path = kagglehub.dataset_download("amananandrai/ag-news-classification-dataset")
print("Path to dataset files:", path)

# 2. List all files in the downloaded directory to find the exact CSV names
print("\n--- Files in Dataset ---")
files = os.listdir(path)
print(files)

# 3. Load the dataset 
# (Assuming the main file is named 'train.csv' or similar based on typical AG News datasets. 
# Update 'train.csv' to the actual filename printed in the step above if it differs.)
file_path = os.path.join(path, 'train.csv')
df = pd.read_csv(file_path)

# 4. Check max rows and columns
print("\n--- Dataset Shape ---")
print(f"Total Rows (Max Rows): {df.shape[0]}")
print(f"Total Columns: {df.shape[1]}")

# 5. Check column names and data types
print("\n--- Data Types & Column Info ---")
# df.info() provides a concise summary including column names, non-null counts, and data types
df.info() 


print("\n--- Some Random Samples ---")
display(df.sample(10))

# %% [markdown] {"jupyter":{"outputs_hidden":false}}
# **EDA (B)**

# %% [code] {"jupyter":{"outputs_hidden":false}}
import matplotlib.pyplot as plt
import seaborn as sns

# Set a clean style for the plots
sns.set_theme(style="whitegrid")

# %% [code] {"jupyter":{"outputs_hidden":false}}
print("--- BEFORE PREPROCESSING ---")

# 1. Check Class Distribution
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='Class Index', palette='viridis')
# sns.countplot(data=df, x='Class Index', hue='Class Index', palette='viridis', legend=False)
plt.title('Class Distribution (Raw Data)')
plt.xlabel('Class Index (1: World, 2: Sports, 3: Business, 4: Sci/Tech)')
plt.ylabel('Number of Articles')
plt.show()

# 2. Check Word Count Distribution
# Temporarily combine Title and Description to get the raw word count
raw_combined = df['Title'].fillna('') + " " + df['Description'].fillna('')
raw_word_counts = raw_combined.apply(lambda x: len(str(x).split()))

plt.figure(figsize=(10, 5))
sns.histplot(raw_word_counts, bins=50, kde=True, color='blue')
plt.title('Word Count Distribution (Raw Data)')
plt.xlabel('Number of Words per Article')
plt.ylabel('Frequency')
# Limit x-axis just in case there are massive outliers
plt.xlim(0, 150) 
plt.show()

# %% [code] {"jupyter":{"outputs_hidden":false}}
# Missing Values
print("Missing values in each column:")
print(df.isnull().sum())

# 2. Count Duplicate Rows
# We check for duplicates where both Title and Description are exactly the same
num_duplicates = df.duplicated(subset=['Title', 'Description']).sum()
print(f"\nTotal duplicate rows in raw data: {num_duplicates}")

# 3. Count URLs
url_pattern = r'http\S+|www\S+'
urls_in_title = df['Title'].str.contains(url_pattern, regex=True, na=False).sum()
urls_in_desc = df['Description'].str.contains(url_pattern, regex=True, na=False).sum()

print(f"\nArticles with URLs in Title: {urls_in_title}")
print(f"Articles with URLs in Description: {urls_in_desc}")

# %% [markdown] {"jupyter":{"outputs_hidden":false}}
# > Data preparation (B)

# %% [markdown] {"jupyter":{"outputs_hidden":false}}
# 
# 
# * **df.info shows 120000 non-null --> This means your raw dataset actually has zero missing values for all 3 columns**
# * **HTML Entity Decoding: It converts artifacts like #39; to ', &amp; to &, and quot; to "**
# * **URL Removal: words with http or www are noise , not actual word=1847 URLs**
# * **Whitespace Normalization**
# * **Duplicates title and descriptions are zero**
# * **re.sub(r"\s+", " ", text) for whitespace normalization with single space**

# %% [markdown] {"jupyter":{"outputs_hidden":false}}
# **Most importantnluy we check the data 120k to 50k training samples**

# %% [code] {"jupyter":{"outputs_hidden":false}}
import os
import re
import json
import pandas as pd
import kagglehub
import html

# Download dataset
path = kagglehub.dataset_download("amananandrai/ag-news-classification-dataset")

# Load both splits
train_raw = pd.read_csv(os.path.join(path, "train.csv"))
test_raw = pd.read_csv(os.path.join(path, "test.csv"))

print(f"Raw train: {train_raw.shape}, Raw test: {test_raw.shape}")

# --- CONFIG ---
SAMPLES_PER_CLASS_TRAIN = 12500  # 12500 x 4 classes = 50,000 total
SAMPLES_PER_CLASS_TEST = 1900    # 1900 x 4 = 7,600 for test
RANDOM_SEED = 42


# --- CLEANING ---
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.strip()
    # text = re.sub(r"#39;", "'", text)
    # text = re.sub(r"&amp;", "&", text)
    # text = re.sub(r'quot;', '"', text)
    text = html.unescape(text)
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def process_split(df, samples_per_class=None):
    df = df.copy()

    # Combine title + description
    df["text"] = df["Title"].apply(clean_text) + " " + df["Description"].apply(clean_text)

    # Shift labels: 1-4 → 0-3 (HuggingFace expects 0-indexed)
    df["label"] = df["Class Index"] - 1

    # Drop bad rows
    df = df.dropna(subset=["text", "label"])
    df = df[df["text"].str.len() > 0]
    df = df.drop_duplicates(subset=["text"])

    # Subsample
    if samples_per_class:
        df = df.groupby("label").apply(
            lambda x: x.sample(n=min(samples_per_class, len(x)), random_state=RANDOM_SEED)
        ).reset_index(drop=True)

    return df[["text", "label"]]


# Process
train_df = process_split(train_raw, samples_per_class=SAMPLES_PER_CLASS_TRAIN)
test_df = process_split(test_raw, samples_per_class=SAMPLES_PER_CLASS_TEST)

print(f"\nProcessed train: {train_df.shape}")
print(f"Processed test: {test_df.shape}")
print(f"\nTrain label distribution:\n{train_df['label'].value_counts().sort_index()}")
print(f"\nTest label distribution:\n{test_df['label'].value_counts().sort_index()}")
print(f"\nSample rows:")
display(train_df.sample(5))

# %% [markdown] {"jupyter":{"outputs_hidden":false}}
# > Saving files for github

# %% [code] {"jupyter":{"outputs_hidden":false}}
id2label = {
    "0": "World",
    "1": "Sports",
    "2": "Business",
    "3": "Sci/Tech"
}

with open("id2label.json", "w") as f:
    json.dump(id2label, f, indent=2)

print("Saved id2label.json")
print(json.dumps(id2label, indent=2))

# %% [code] {"jupyter":{"outputs_hidden":false}}
# Save data_prep.py as a standalone script file

data_prep_code = '''import os
import re
import json
import pandas as pd
try:
    import kagglehub
except ImportError as e:
    raise ImportError(
        "The 'kagglehub' module was not found. "
        "If you are running this script outside of Kaggle, please install it by running: pip install kagglehub"
    ) from e

SAMPLES_PER_CLASS_TRAIN = 12500
SAMPLES_PER_CLASS_TEST = 1900
RANDOM_SEED = 42


import html

def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    text = text.strip()
    
    # 1. Safely and completely unescape ALL HTML entities
    # text = re.sub(r"#39;", "'", text)
    # text = re.sub(r"&amp;", "&", text)
    # text = re.sub(r'quot;', '"', text)
    text = html.unescape(text)
    
    # 2. Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)
    
    # 3. Normalize whitespace
    text = re.sub(r"\s+", " ", text)
    
    return text.strip()


def process_split(df, samples_per_class=None):
    df = df.copy()
    df["text"] = df["Title"].apply(clean_text) + " " + df["Description"].apply(clean_text)
    df["label"] = df["Class Index"] - 1
    df = df.dropna(subset=["text", "label"])
    df = df[df["text"].str.len() > 0]
    df = df.drop_duplicates(subset=["text"])
    if samples_per_class:
        df = df.groupby("label").apply(
            lambda x: x.sample(n=min(samples_per_class, len(x)), random_state=RANDOM_SEED)
        ).reset_index(drop=True)
    return df[["text", "label"]]


def main():
    path = kagglehub.dataset_download("amananandrai/ag-news-classification-dataset")
    train_raw = pd.read_csv(os.path.join(path, "train.csv"))
    test_raw = pd.read_csv(os.path.join(path, "test.csv"))

    train_df = process_split(train_raw, SAMPLES_PER_CLASS_TRAIN)
    test_df = process_split(test_raw, SAMPLES_PER_CLASS_TEST)

    id2label = {"0": "World", "1": "Sports", "2": "Business", "3": "Sci/Tech"}
    with open("id2label.json", "w") as f:
        json.dump(id2label, f, indent=2)

    os.makedirs("data", exist_ok=True)
    train_df.to_csv("data/train.csv", index=False)
    test_df.to_csv("data/test.csv", index=False)
    print(f"Train: {len(train_df)}, Test: {len(test_df)}")


if __name__ == "__main__":
    main()
'''

os.makedirs("src", exist_ok=True)
with open("src/data_prep.py", "w") as f:
    f.write(data_prep_code)

print(" Saved src/data_prep.py")

# %% [code] {"jupyter":{"outputs_hidden":false}}
os.makedirs("data", exist_ok=True)
train_df.to_csv("data/train.csv", index=False)
test_df.to_csv("data/test.csv", index=False)
print("Saved data/train.csv and data/test.csv")

# %% [markdown] {"jupyter":{"outputs_hidden":false}}
# **EDA after preprocessing (B)**

# %% [code] {"jupyter":{"outputs_hidden":false}}
print("AFTER PREPROCESSING")

# 1. Check Class Distribution After Cleaning
plt.figure(figsize=(8, 5))
sns.countplot(data=train_df, x='label',palette='viridis')

plt.title('Class Distribution (Cleaned Data)')
plt.xlabel('Label (0: World, 1: Sports, 2: Business, 3: Sci/Tech)')
plt.ylabel('Number of Articles')
plt.show()

# 2. Check Word Count Distribution After Cleaning
clean_word_counts = train_df['text'].apply(lambda x: len(str(x).split()))

plt.figure(figsize=(10, 5))
sns.histplot(clean_word_counts, bins=50, kde=True, color='orange')
plt.title('Word Count Distribution (Cleaned Data)')
plt.xlabel('Number of Words per Article')
plt.ylabel('Frequency')
plt.xlim(0, 150)
plt.show()

# %% [code] {"jupyter":{"outputs_hidden":false}}
url_pattern = r'http\S+|www\S+'

# Count how many rows in the cleaned 'text' column still contain a URL
urls_in_cleaned_text = train_df['text'].str.contains(url_pattern, regex=True, na=False).sum()
print(f"Articles with URLs in cleaned text: {urls_in_cleaned_text}")